In [0]:
# =============================================================================
# Notebook: 05_simulate_new_batch
# Purpose : Simulate a new incoming data batch containing NEW orders
#           and UPDATES to existing orders. This lets us
#           test the CDC MERGE logic in 04_bronze_to_silver_cdc.
# =============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
import random
from datetime import datetime, timedelta

random.seed(99)  # different seed - simulates a "new" batch, not identical data

storage_account = "stretailcdcproj"
bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/retail_orders/"

df_existing = spark.read.format("delta").load(bronze_path)

# ---- Part 1: Simulate UPDATES to existing orders ----
# Take 50 random existing order_ids and change their amount + last_modified
existing_sample = df_existing.orderBy(F.rand(seed=1)).limit(50).collect()

updated_rows = []
now = datetime.now()
for row in existing_sample:
    new_amount = round(random.uniform(5, 500), 2)
    updated_rows.append((
        row["order_id"], row["customer_id"], row["product"], row["region"],
        row["quantity"], new_amount, row["order_date"], now
    ))

# ---- Part 2: Simulate NEW orders (order_id continues from 5000) ----
products = ['Shampoo', 'Conditioner', 'Face Cream', 'Lipstick', 'Serum', 'Sunscreen']
regions = ['North', 'South', 'East', 'West']

new_rows = []
for i in range(5001, 5101):  # 100 new orders
    new_rows.append((
        i,
        random.randint(1000, 2000),
        random.choice(products),
        random.choice(regions),
        random.randint(1, 5),
        round(random.uniform(5, 500), 2),
        now,
        now
    ))

schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("region", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("order_date", TimestampType(), True),
    StructField("last_modified", TimestampType(), True),
])

df_new_batch = spark.createDataFrame(updated_rows + new_rows, schema)

# ---- Append this new batch into Bronze (simulating new data arriving) ----
df_new_batch.write.mode("append").format("delta").save(bronze_path)

print(f"Simulated batch added: {len(updated_rows)} updated + {len(new_rows)} new orders")
print(f"Total rows in bronze now: {spark.read.format('delta').load(bronze_path).count()}")


✅ Simulated batch added: 50 updated + 100 new orders
Total rows in bronze now: 5170
